# Avant le TP6 : lire une métrique, de l'agent JMX jusqu'à l'alerte

Le TP6 interroge la chaîne de supervision du lab depuis le notebook. Cette démonstration suit une seule métrique de la JVM, le nombre de threads, à chaque maillon :

1. telle que l'agent JMX d'un nœud l'expose, en texte ;
2. telle que Prometheus la renvoie à une requête PromQL, en JSON ;
3. puis les règles d'alerte et leur état, telles que Prometheus les publie.

Sorties de référence : exécution du 07/11/2026 22:41 UTC sur le lab (mode `extended`).

In [1]:
import requests

PROMETHEUS = "http://prometheus:9090"

## 1. L'agent JMX : du texte

Chaque nœud publie ses métriques sur le port 9404. Une métrique, c'est une ligne `HELP` (description), une ligne `TYPE` (`gauge` pour une valeur instantanée, `counter` pour un total qui ne fait que croître), puis une ligne par série : `nom{etiquettes} valeur`.

In [2]:
texte = requests.get("http://kafka-2:9404/metrics", timeout=10).text
for ligne in texte.splitlines():
    if "jvm_threads_current" in ligne:
        print(ligne)
print(len(texte.splitlines()), "lignes au total")

# HELP jvm_threads_current Current thread count of a JVM
# TYPE jvm_threads_current gauge
jvm_threads_current 111.0
6771 lignes au total


## 2. Prometheus : du JSON

Prometheus collecte ces pages toutes les 15 s et ajoute les étiquettes `job` et `instance`. Son API répond aux requêtes PromQL. La valeur arrive sous forme de chaîne, avec son horodatage.

In [3]:
reponse = requests.get(PROMETHEUS + "/api/v1/query", params={"query": "jvm_threads_current"}, timeout=10).json()
print(reponse["status"], reponse["data"]["resultType"])
for resultat in reponse["data"]["result"]:
    print(resultat["metric"]["job"], resultat["metric"]["instance"], resultat["value"])

success vector
kafka kafka-3 [1794091329.306, '113']
kafka-clients loadgen-consumer [1794091329.306, '13']
kafka kafka-1 [1794091329.306, '111']
kafka kafka-2 [1794091329.306, '111']
kafka-clients loadgen-producer [1794091329.306, '13']


Une requête peut agréger : ici, le total des threads par job.

In [4]:
reponse = requests.get(PROMETHEUS + "/api/v1/query", params={"query": "sum by (job) (jvm_threads_current)"},
                       timeout=10).json()
for resultat in reponse["data"]["result"]:
    print(resultat["metric"]["job"], float(resultat["value"][1]))

kafka 335.0
kafka-clients 26.0


## 3. Les règles d'alerte

Prometheus évalue ses règles toutes les 15 s. Chaque règle a un état : `inactive`, `pending` (condition vraie, délai `for` pas encore écoulé) ou `firing`.

In [5]:
regles = requests.get(PROMETHEUS + "/api/v1/rules", timeout=10).json()["data"]["groups"]
for groupe in regles:
    for regle in groupe["rules"]:
        print("%-25s %-40s %s" % (groupe["name"], regle["name"], regle["state"]))

kafka-cluster             KafkaNodeDown                            inactive
kafka-cluster             KafkaNoActiveController                  inactive
kafka-cluster             KafkaOfflinePartitions                   inactive
kafka-cluster             KafkaUnderMinIsrPartitions               inactive
kafka-cluster             KafkaUnderReplicatedPartitions           inactive
kafka-cluster             KafkaFencedBrokers                       inactive
kafka-cluster             KafkaRequestHandlerSaturated             inactive
kafka-consumers           KafkaConsumerGroupLagHigh                inactive
kafka-storage             KafkaBrokerLogSizeHigh                   inactive


## La méthode du TP6

Ouvrez `tps/jour2/tp6_observabilite/enonce.ipynb`.

- Vous écrirez les fonctions qui lisent chaque maillon : `metriques()` pour l'agent, `promql()` pour Prometheus, `etats_alertes()` pour les alertes. Elles servent ensuite jusqu'à la fin du TP.
- Gardez Grafana ouvert dans le navigateur (http://localhost:3000) : chaque valeur calculée dans le notebook a sa tuile.
- Deux cellules attendent une alerte (6.7 et 6.8) : 2 à 4 minutes en exécution, à ne pas interrompre.
- Le lab doit être en mode `extended`.